In [55]:
from rich.console import Console
from dotenv import load_dotenv
from openai import OpenAI
import json
load_dotenv(override=True)

openai = OpenAI()

In [56]:
def show(text):
    try:
        Console().print(text)
    except:
        print(text)

In [57]:
def get_todo_report() -> str:
    """Generate a report of TODO comments in the code."""
    result = ""
    for index, todo in enumerate(todos):
        if completed[index]:
            result += f"Todo #{index + 1}. [green][strike]{todo}[/strike][/green]\n"
        else:
            result += f"Todo #{index + 1}. {todo}\n"
    show(result)
    return result

In [58]:
create_todos_json = {
    "name": "create_todos",
    "description": "Add new todos from a list of descriptions and return the full list",
    "parameters": {
        "type": "object",
        "properties": {
            "descriptions": {
                'type': 'array',
                'items': {'type': 'string'},
                'title': 'Descriptions'
                }
            },
        "required": ["descriptions"],
        "additionalProperties": False
    }
}

mark_complete_json = {
    "name": "mark_complete",
    "description": "Mark complete the todo at the given position (starting from 1) and return the full list",
    "parameters": {
        'properties': {
            'index': {
                'description': 'The 1-based index of the todo to mark as complete',
                'title': 'Index',
                'type': 'integer'
                },
            'completion_notes': {
                'description': 'Notes about how you completed the todo in rich console markup',
                'title': 'Completion Notes',
                'type': 'string'
                }
            },
        'required': ['index', 'completion_notes'],
        'type': 'object',
        'additionalProperties': False
    }
}

In [59]:
tools = [{"type": "function", "function": create_todos_json},
        {"type": "function", "function": mark_complete_json}]

In [60]:
def create_todos(descriptions: list[str]) -> str:
    todos.extend(descriptions)
    completed.extend([False] * len(descriptions))
    return get_todo_report()

def mark_complete(index: int, completion_notes: str) -> str:
    if index > 0 and index <= len(todos):
        completed[index - 1] = True
    else:
        return "No Todo at this index"
    return get_todo_report()

def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        print(f"here is tool name {tool_name}")
        arguments = json.loads(tool_call.function.arguments)
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else {}
        results.append({"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id})
    return results

In [61]:
def loop(messages):
    done = False
    while not done:
      Console().print(f"** Here are the messages *** [red]{messages}[]/red")
      Console().print(f"** Here are the tools *** [blue]{tools}[/blue]")
      response = openai.chat.completions.create(model="gpt-5.2", messages=messages, tools=tools, reasoning_effort="none")
      finish_reason = response.choices[0].finish_reason
      if finish_reason == "tool_calls":
        message = response.choices[0].message
        tool_calls = message.tool_calls
        results = handle_tool_calls(tool_calls)
        messages.append(message)
        messages.extend(results)
        print(f"** Here is the LLM message *** {message}")
        print(f"** Here are the results *** {results}")
        print(f"** Here are the tool calls *** {tool_calls}")
      else:
        done = True
    show(response.choices[0].message.content)

In [62]:
system_message = """
You are given a problem to solve, by using your todo tools to plan a list of steps, then carrying out each step in turn.
Now use the todo list tools, create a plan, carry out the steps, and reply with the solution.
If any quantity isn't provided in the question, then include a step to come up with a reasonable estimate.
Provide your solution in Rich console markup without code blocks.
Do not ask the user questions or clarification; respond only with the answer after using your tools.
"""
user_message = """"
Given a name Nayan, provide an innovative greeting for work anniversary.
Find out if the name is palindrome.
Share the result along with the greeting message.
"""
messages = [{"role": "system", "content": system_message}, {"role": "user", "content": user_message}]

In [63]:
todos, completed = [], []
loop(messages)

** Here are the messages *** [{'role': 'system', 'content': "\nYou are given a problem to solve, by using your todo
tools to plan a list of steps, then carrying out each step in turn.\nNow use the todo list tools, create a plan, 
carry out the steps, and reply with the solution.\nIf any quantity isn't provided in the question, then include a 
step to come up with a reasonable estimate.\nProvide your solution in Rich console markup without code blocks.\nDo 
not ask the user questions or clarification; respond only with the answer after using your tools.\n"}, {'role': 
'user', 'content': '"\nGiven a name Nayan, provide an innovative greeting for work anniversary.\nFind out if the 
name is palindrome.\nShare the result along with the greeting message.\n'}][]/red

** Here are the tools *** [{'type': 'function', 'function': {'name': 'create_todos', 'description': 'Add new todos 
from a list of descriptions and return the full list', 'parameters': {'type': 'object', 'properties': 
{'descriptions': {'type': 'array', 'items': {'type': 'string'}, 'title': 'Descriptions'}}, 'required': 
['descriptions'], 'additionalProperties': False}}}, {'type': 'function', 'function': {'name': 'mark_complete', 
'description': 'Mark complete the todo at the given position (starting from 1) and return the full list', 
'parameters': {'properties': {'index': {'description': 'The 1-based index of the todo to mark as complete', 
'title': 'Index', 'type': 'integer'}, 'completion_notes': {'description': 'Notes about how you completed the todo 
in rich console markup', 'title': 'Completion Notes', 'type': 'string'}}, 'required': ['index', 
'completion_notes'], 'type': 'object', 'additionalProperties': False}}}]

here is tool name create_todos


Todo #1. Draft an innovative work-anniversary greeting tailored to the name Nayan.
Todo #2. Check whether the name "Nayan" is a palindrome (case-insensitive).
Todo #3. Combine greeting and palindrome result into a single Rich console markup response.

** Here is the LLM message *** ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_XBbFcNhr5XO5jw3xmUjp7wHO', function=Function(arguments='{"descriptions":["Draft an innovative work-anniversary greeting tailored to the name Nayan.","Check whether the name \\"Nayan\\" is a palindrome (case-insensitive).","Combine greeting and palindrome result into a single Rich console markup response."]}', name='create_todos'), type='function')])
** Here are the results *** [{'role': 'tool', 'content': '"Todo #1. Draft an innovative work-anniversary greeting tailored to the name Nayan.\\nTodo #2. Check whether the name \\"Nayan\\" is a palindrome (case-insensitive).\\nTodo #3. Combine greeting and palindrome result into a single Rich console markup response.\\n"', 'tool_call_id': 'call_XBbFcNhr5XO5jw3xmUjp7wHO'}]
** Here are the tool calls *** [ChatCompletionMessageFunctionToolCal

...

Todo #1. Draft an innovative work-anniversary greeting tailored to the name Nayan.
Todo #2. Check whether the name "Nayan" is a palindrome (case-insensitive).
Todo #3. Combine greeting and palindrome result into a single Rich console markup response.

** Here is the LLM message *** ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_pXpOUHVeC13mYlN4GGpXnhAr', function=Function(arguments='{"index":3,"completion_notes":"Merged the greeting and the palindrome finding into one concise message with Rich markup headings and emphasis."}', name='mark_complete'), type='function')])
** Here are the results *** [{'role': 'tool', 'content': '"Todo #1. [green][strike]Draft an innovative work-anniversary greeting tailored to the name Nayan.[/strike][/green]\\nTodo #2. [green][strike]Check whether the name \\"Nayan\\" is a palindrome (case-insensitive).[/strike][/green]\\nTodo #3. [green][strike]Combine greeting and palindrome result into a single Rich console markup response.[/strike][/green]\\n"', 'tool_call_id': 'call_pXpOUHVeC13mYlN4GGpXnhAr'}]
** Here are the tool calls *** [ChatCompletionMessageFunctionToolCall(id='call_

** Here are the messages *** [{'role': 'system', 'content': "\nYou are given a problem to solve, by using your todo
tools to plan a list of steps, then carrying out each step in turn.\nNow use the todo list tools, create a plan, 
carry out the steps, and reply with the solution.\nIf any quantity isn't provided in the question, then include a 
step to come up with a reasonable estimate.\nProvide your solution in Rich console markup without code blocks.\nDo 
not ask the user questions or clarification; respond only with the answer after using your tools.\n"}, {'role': 
'user', 'content': '"\nGiven a name Nayan, provide an innovative greeting for work anniversary.\nFind out if the 
name is palindrome.\nShare the result along with the greeting message.\n'}, ChatCompletionMessage(content=None, 
refusal=None, role='assistant', annotations=[], audio=None, function_call=None, 
tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_XBbFcNhr5XO5jw3xmUjp7wHO', 
function=Function(arguments='{"descriptions":["Draft an innovative work-anniversary greeting tailored to the name 
Nayan.","Check whether the name \\"Nayan\\" is a palindrome (case-insensitive).","Combine greeting and palindrome 
result into a single Rich console markup response."]}', name='create_todos'), type='function')]), {'role': 'tool', 
'content': '"Todo #1. Draft an innovative work-anniversary greeting tailored to the name Nayan.\\nTodo #2. Check 
whether the name \\"Nayan\\" is a palindrome (case-insensitive).\\nTodo #3. Combine greeting and palindrome result 
into a single Rich console markup response.\\n"', 'tool_call_id': 'call_XBbFcNhr5XO5jw3xmUjp7wHO'}, 
ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None,
tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_OUHrb9xjMq8lIqyMNL9kSbuh', 
function=Function(arguments='{"index":1,"completion_notes":"Created an original, work-appropriate anniversary 
greeting using a wordplay nod to the name (\\"Nayan\\" ~ \\"nayan\\" meaning \\"eyes/vision\\" in some contexts) 
and a forward-looking theme."}', name='mark_complete'), type='function')]), {'role': 'tool', 'content': '"Todo #1. 
Draft an innovative work-anniversary greeting tailored to the name Nayan.\\nTodo #2. Check whether the name 
\\"Nayan\\" is a palindrome (case-insensitive).\\nTodo #3. Combine greeting and palindrome result into a single 
Rich console markup response.\\n"', 'tool_call_id': 'call_OUHrb9xjMq8lIqyMNL9kSbuh'}, 
ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None,
tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_68BLdbBdrg1dfe7eeUEcMOpk', 
function=Function(arguments='{"index":2,"completion_notes":"Checked palindrome by reversing letters: Nayan → n a y 
a n; reversed is n a y a n. Case-insensitive match holds, so it is a palindrome."}', name='mark_complete'), 
type='function')]), {'role': 'tool', 'content': '"Todo #1. Draft an innovative work-anniversary greeting tailored 
to the name Nayan.\\nTodo #2. Check whether the name \\"Nayan\\" is a palindrome (case-insensitive).\\nTodo #3. 
Combine greeting and palindrome result into a single Rich console markup response.\\n"', 'tool_call_id': 
'call_68BLdbBdrg1dfe7eeUEcMOpk'}, ChatCompletionMessage(content=None, refusal=None, role='assistant', 
annotations=[], audio=None, function_call=None, 
tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_pXpOUHVeC13mYlN4GGpXnhAr', 
function=Function(arguments='{"index":3,"completion_notes":"Merged the greeting and the palindrome finding into one
concise message with Rich markup headings and emphasis."}', name='mark_complete'), type='function')]), {'role': 
'tool', 'content': '"Todo #1. Draft an innovative work-anniversary greeting tailored to the name Nayan.\\nTodo #2. 
Check whether the name \\"Nayan\\" is a palindrome (case-insensitive).\\nTodo #3. Combine greeting and palindrome 
result into a single Rich console markup

** Here are the tools *** [{'type': 'function', 'function': {'name': 'create_todos', 'description': 'Add new todos 
from a list of descriptions and return the full list', 'parameters': {'type': 'object', 'properties': 
{'descriptions': {'type': 'array', 'items': {'type': 'string'}, 'title': 'Descriptions'}}, 'required': 
['descriptions'], 'additionalProperties': False}}}, {'type': 'function', 'function': {'name': 'mark_complete', 
'description': 'Mark complete the todo at the given position (starting from 1) and return the full list', 
'parameters': {'properties': {'index': {'description': 'The 1-based index of the todo to mark as complete', 
'title': 'Index', 'type': 'integer'}, 'completion_notes': {'description': 'Notes about how you completed the todo 
in rich console markup', 'title': 'Completion Notes', 'type': 'string'}}, 'required': ['index', 
'completion_notes'], 'type': 'object', 'additionalProperties': False}}}]

Work Anniversary Greeting for Nayan  
Happy Work Anniversary, Nayan! Your journey here has been all about vision and forward momentum—like your name, you
bring clarity to what matters and keep progress moving in the right direction. Here’s to the impact you’ve made so 
far, and to an even brighter year ahead.

Palindrome Check  
Name: Nayan  
Result: Yes—it's a palindrome. (It reads the same forward and backward: nayan.)